# Tokenizatorul, modelul și generarea

Pașii pe care îi ascunde pipeline, făcuți de mână cu Transformers. Tokenizarea, încărcarea modelului, șablonul de chat, parametrii generării și o conversație afișată pe măsură ce este generată.

Codul din acest notebook vine de pe pagina [Tokenizatorul, modelul și generarea](https://inteligentartificiala.ro/laboratoare/model-local-transformers/tokenizator-si-generare/) a laboratorului [Un model de limbaj local cu Transformers](https://inteligentartificiala.ro/laboratoare/model-local-transformers/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro). Explicațiile fiecărui pas sunt pe pagină.

În Google Colab, codul rulează mult mai repede pe o placă grafică. Dacă nu este deja aleasă, se alege din meniul **Runtime**, opțiunea **Change runtime type**, o variantă cu GPU. Celulele se rulează în ordine, de sus în jos.

In [ ]:
%pip install -q -U transformers accelerate

## Tokenizatorul

Modelul nu vede litere, ci numere. **Tokenizatorul** (_tokenizer_) taie textul în tokenuri și înlocuiește fiecare token cu ID-ul lui din vocabular. Clasele `Auto…` aleg automat implementarea potrivită pentru modelul cerut.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")

ids = tokenizer("Transformers runs language models locally.")["input_ids"]
print("ID-uri:  ", ids)
print("Tokenuri:", tokenizer.convert_ids_to_tokens(ids))
print("Înapoi:  ", tokenizer.decode(ids))
print("Mărimea vocabularului:", len(tokenizer))

## Modelul și șablonul de chat

Codul de mai jos încarcă modelul, construiește promptul unei conversații și generează un răspuns.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
print(f"Dispozitiv: {model.device}, tip de date: {model.dtype}, {model.num_parameters():,} parametri")

messages = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "Explain in two sentences what a token is."},
]
# Șablonul de chat transformă lista de mesaje în textul exact pe care l-a văzut modelul la antrenare.
print(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
).to(model.device)
output = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.2, top_p=0.9)

# Rezultatul conține și promptul, așa că păstrăm doar tokenurile noi.
new_tokens = output[0, inputs["input_ids"].shape[1]:]
print(tokenizer.decode(new_tokens, skip_special_tokens=True))

## O conversație afișată pe măsura generării

Asistenții conversaționali afișează textul pe măsură ce apare, nu tot deodată, la final. În Transformers, acest lucru se face cu un `TextStreamer`. Exemplul de mai jos poartă și o conversație cu două replici.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TextStreamer

model_id = "HuggingFaceTB/SmolLM2-360M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype="auto", device_map="auto")
# Afișează fiecare bucată de text imediat ce este generată.
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

messages = []
for question in ["Name three planets.", "Which of them is the largest?"]:
    print(f"\n> {question}")
    messages.append({"role": "user", "content": question})
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(model.device)
    output = model.generate(**inputs, max_new_tokens=80, streamer=streamer)
    # Modelul nu ține minte conversația, așa că răspunsul lui intră în istoricul trimis data viitoare.
    answer = tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    messages.append({"role": "assistant", "content": answer})